Please click the "Play" button below to start the *inference server*. TODO: write what an inference server is/what it is needed for.

In [1]:
! docker compose up -d

WARN[0000] /home/cewit_admin/hack/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 
[+] up 1/1
 ✔ Container inference-server Running                                       0.0s
[+] up 1/1
 ✔ Container inference-server Running                                       0.0s


Note that even though Docker says inference-server is running, it may take a minute or two to start!

In [2]:
# PLEASE ADD YOUR ROBOFLOW API KEY BELOW!
ROBOFLOW_API_KEY="AaTSzksKDzavqADpqy4n"

In [ ]:
import cv2
import os
from inference_sdk import InferenceHTTPClient

# 1. Initialize the client
client = InferenceHTTPClient(
    api_url="http://localhost:9001",
    api_key=ROBOFLOW_API_KEY
)
print("Checking model availability...")
try:
    client.select_model("yolov8n-640/1") # Replace with your REAL ID
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
# 0 is usually the default built-in webcam
cap = cv2.VideoCapture(0)

while True:
    # Capture frame-by-frame
    ret, frame = cap.read()
    
    if not ret:
        print("Failed to grab frame")
        break

    # 2. Run Inference
    # Replace "your-model-id/version" with your actual project ID and version number
    # Example: "yolov8n-640/1"
    results = client.infer(frame, model_id="yolov8n-640")

    # 3. Parse and Draw Predictions
    # The response is a dictionary containing a list of predictions
    for prediction in results.get("predictions", []):
        x = int(prediction["x"])
        y = int(prediction["y"])
        w = int(prediction["width"])
        h = int(prediction["height"])
        label = prediction["class"]
        confidence = prediction["confidence"]

        # Calculate bounding box coordinates (Roboflow returns center x,y)
        x1 = int(x - w / 2)
        y1 = int(y - h / 2)
        x2 = int(x + w / 2)
        y2 = int(y + h / 2)

        # Draw the rectangle
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Draw the label
        text = f"{label} {confidence:.2f}"
        cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display the resulting frame
    cv2.imshow('Live Webcam Feed', frame)

    # Stop the loop when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the capture and close windows
cap.release()
cv2.destroyAllWindows()

Checking model availability...
Model loaded successfully!


KeyboardInterrupt: 

: 